# Wan2.1 — Job `neon_city_walk` (T2V)

**Spec:** 5 s • 480p (832×480) • 16 fps nativo (24 opcional via ffmpeg) • câmera acompanhando o personagem

**GPU no Colab (Runtime → Change runtime type):**
- **t2v-1.3B** (padrão deste notebook): qualquer GPU — T4 16 GB, L4, V100, A100. Usa ~8,2 GB de VRAM; expecte ~20–40 min por clipe de 5 s num T4.
- **t2v-14B** (realismo bem superior): A100 40 GB+ — de preferência o **A100 High-RAM** (Colab Pro+), pois com `offload_model` o modelo ocupa ~30 GB de RAM.

Rode as células em ordem. O vídeo sai em `/content/wan2.1/` e é exibido na última célula.


In [ ]:
# O Colab já traz torch com CUDA — NÃO reinstale torch.
# flash_attn é pulado de propósito: é opcional (o código cai no fallback SDPA do PyTorch)
# e compilar no Colab leva 30+ min e costuma quebrar.
!pip install -q \
    "diffusers>=0.31.0" "transformers>=4.49.0" "tokenizers>=0.20.3" \
    "accelerate>=1.1.1" "opencv-python>=4.9.0.80" "tqdm" \
    "imageio" "imageio-ffmpeg" "easydict" "ftfy" "numpy>=1.23.5,<2" \
    huggingface_hub

In [ ]:
# Clona o repositório (branch com o job + flag --neg_prompt)
!test -d /content/wan2.1 || git clone -b arena/01a04839-wan2-1 https://github.com/anonyby777-lgtm/wan2.1.git /content/wan2.1
%cd /content/wan2.1

In [ ]:
# ── Checkpoint Wan2.1-T2V-1.3B (≈14,5 GB: DiT 2,5 GB + T5 11 GB + VAE 0,3 GB) ──
!huggingface-cli download Wan-AI/Wan2.1-T2V-1.3B --local-dir ./Wan2.1-T2V-1.3B

# ── Alternativa: Wan2.1-T2V-14B (≈40 GB — só com A100 40 GB+ e RAM de sobra) ──
# !huggingface-cli download Wan-AI/Wan2.1-T2V-14B --local-dir ./Wan2.1-T2V-14B

In [ ]:
# ── Geração (padrão: t2v-1.3B) ──
# Usou o checkpoint 14B? Troque para:  !MODEL=t2v-14B SEED=42 bash jobs/neon_city_walk/run.sh
# Para 24 fps:  !MODEL=t2v-1.3B SEED=42 SAVE_FPS=24 bash jobs/neon_city_walk/run.sh
!MODEL=t2v-1.3B SEED=42 bash jobs/neon_city_walk/run.sh

In [ ]:
import glob, os
from IPython.display import Video

files = sorted(glob.glob('t2v-*.mp4'), key=os.path.getmtime)
assert files, "Nenhum .mp4 gerado ainda — rode a célula de geração."
display(Video(files[-1], embed=True))
print("Arquivo:", os.path.abspath(files[-1]))

## Iterar

- **Nova variação:** mude `SEED` (ex.: `SEED=7`) e rode de novo — o checkpoint fica em disco, só roda a geração.
- **Prompt/negative prompt:** edite `jobs/neon_city_walk/PROMPT.txt` e `NEGATIVE_PROMPT.txt` (ou passe outra string via `--prompt`/`--neg_prompt`).
- **1.3B** usa automaticamente `--sample_shift 8 --sample_guide_scale 6` (recomendação oficial); **14B** usa o padrão 5,0/5,0.
- O `run.sh` já aplica `--offload_model True --t5_cpu` (T5 na RAM) — importante no Colab.
- Se o runtime reiniciar, o `/content` sobrevive à sessão: só re-instale as libs (célula 1) e re-rodar a geração.
